## Загрузка данных

In [ ]:
import math
import os
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
df = pd.read_csv(os.path.join('..', 'output', 'receipt.csv.gz'), compression='gzip')
df['date'] = pd.to_datetime(df['date'])
figsize=(30, 18)
names = df['row'].unique().tolist()
d = { name: df[(df['row'] == name) & (df['col'] == 'amount')][['date', 'value']] for name in names }

In [ ]:
cols = int(math.sqrt(len(names)))
rows = (len(names) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=figsize)
for k, name in enumerate(names):
  f = d[name]
  # Группируем данные по месяцам и усредняем значения
  f_monthly = f.resample('MS', on='date').mean()
  y, x = divmod(k, cols)
  ax = axes[y, x]
  # Строим бары по агрегированным данным  
  ax.bar(f_monthly.index, f_monthly['value'], width=25)  # width — ширина бара в днях    
  ax.set_title(name)
  ax.xaxis.set_major_locator(mdates.MonthLocator(interval=12)) # Деления каждый год
  ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))  # Минорные деления раз в месяц
  ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))  # Скрываем подписи на основных делениях
  ax.xaxis.set_minor_formatter(mdates.DateFormatter(''))       # Без подписей
plt.tight_layout()
plt.show()

## Все на одном графике. Легенда отсортирована по максимальному значению.

In [ ]:
o = [(d[name]['value'].max(), name) for name in names]
o.sort(reverse = True)
plt.figure(figsize=figsize)
for _, name in o:
  f = df[(df['row'] == name) & (df['col'] == 'amount')][['date', 'value']]
  plt.plot(f['date'], f['value'], label=name, linewidth=2)
plt.xlabel('Дата')
plt.ylabel('Cумма (руб.)')
plt.legend()
plt.show()

In [ ]:
g = df[df['col'] == 'amount'][['row', 'value']].groupby('row').sum()
g.plot.pie(y = 'value', figsize = figsize)
plt.title('По всем данным')
plt.ylabel('') # Убираем метку оси Y (она не нужна для pie‑диаграммы)
plt.show()